# 02 — Baseline texte : RoBERTa AFC + AFD

Ce notebook entraîne et évalue les modèles **RoBERTa-base** de référence sur les deux tâches texte : **AFC** (macro F1, 6 classes) et **AFD** (binary F1, détection de phrases fallacieuses). Le protocole d'évaluation est une cross-validation leave-one-dialogue-out sur **35 folds**. Les modèles ont été entraînés via `scripts/run_roberta_afc.py` et `scripts/run_roberta_afd.py` ; les résultats sont chargés depuis `results/results.json`.

## Sommaire

- [Note architecture MAMKit](#note--utilisation-de-mamkit-pour-)
- [Run AFC (35 folds LODO + BM3)](#run-reproductible-code-projet-dans-srcexperiments)
- [Run AFD (35 folds LODO + BM3)](#afd--détection-de-fallacies-dans-les-arguments-binaire-niveau-phrase)
- [Résultats AFC & AFD — tableau comparatif](#afc--afd--résultats-et-interprétation)

In [12]:
import sys
from pathlib import Path

try:
    ROOT = Path(globals()['__vsc_ipynb_file__']).resolve().parent.parent
except KeyError:
    ROOT = Path.cwd()
    for _ in range(4):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.configs.text_configs import get_roberta_afc_config, get_roberta_afd_config
from src.experiments.mmused_text import (
    DEFAULT_DATA_PATH,
    DEFAULT_RESULTS_PATH,
    make_mmused_fallacy_loader,
    prepare_text_reproducibility,
    run_mmused_text_cv,
)
from src.utils.results import ResultsManager

DATA_PATH = DEFAULT_DATA_PATH
RESULTS_PATH = DEFAULT_RESULTS_PATH


In [13]:
# Seeds + HF cache + matmul precision (call once per kernel restart)
prepare_text_reproducibility(seed=42)


Seed set to 42


# NOTE : Utilisation de MAMKit pour :
- Chargement des données : MMUSEDFallacy, get_splits('mancini-et-al-2024')
- Modèle : mamkit.models.text.Transformer
- Collators : TextTransformerCollator, UnimodalCollator
- Boucle d'entraînement : MAMKitLightingModel

La config MAMKit 'mancini-et-al-2024' a un bug connu :
 TypeError: BaseConfig.__init__() missing required argument: 'seeds'
On définit les hyperparamètres manuellement en suivant l'architecture du papier,
avec des poids de classes calculés depuis notre distribution de données.

In [14]:
# TRANSFORMERS_CACHE is set in prepare_text_reproducibility() (cell above).


### Run reproductible (code projet dans `src/experiments`)

Tout cela utilise encore **MAMKit** (`MMUSEDFallacy`, splits, modèles via `TextTrainer`). Lancer les **loaders** (cellule suivante), puis **AFC** (CV 35-fold + BM3), puis **AFD** (même protocole CV ; **beaucoup plus lent** — ~17k phrases).

In [15]:
# Build loaders once; training cells reuse `loader_afc` / `loader_afd`.
loader_afc = make_mmused_fallacy_loader("afc")
loader_afd = make_mmused_fallacy_loader("afd")

print(f"AFC: {loader_afc.data.shape}")
print(f"AFD: {loader_afd.data.shape}")


Building AFC Context: 100%|██████████| 1278/1278 [00:00<00:00, 17052.46it/s]


AFC: (1278, 8)


Building AFD data...: 100%|██████████| 35/35 [00:01<00:00, 33.13it/s]


AFD: (17118, 7)


### Exécuter AFC en arrière-plan (résiste aux déconnexions SSH)

Le **kernel du notebook** peut s'arrêter si la session distante se coupe. Pour un long CV + BM3, utiliser la **même logique** depuis un terminal :

```bash
cd /mm_argfallacy
mkdir -p logs
nohup ./mmarg_env/bin/python scripts/run_roberta_afc.py > logs/roberta_afc.log 2>&1 &
tail -f logs/roberta_afc.log   # optionnel : surveiller la progression ; Ctrl+C arrête tail uniquement
```

**tmux** (se rattacher après reconnexion : `tmux attach -t roberta_afc`) : ouvrir une session, lancer `./mmarg_env/bin/python scripts/run_roberta_afc.py`, puis détacher avec **Ctrl+B** puis **D**.

In [ ]:
# AFC — full leave-one-dialogue-out CV (35 folds) + BM3 refit (3 checkpoints on disk).
summary_afc = run_mmused_text_cv(
    get_roberta_afc_config(),
    loader=loader_afc,
    save_bm3_checkpoints_after=True,
)
print(summary_afc)

# Smoke test (uncomment):
# summary_afc = run_mmused_text_cv(
#     get_roberta_afc_config(),
#     loader=loader_afc,
#     max_folds=3,
#     save_bm3_checkpoints_after=False,
# )


### AFD — Détection de Fallacies dans les Arguments (binaire, niveau phrase)

Même protocole CV (**35 folds**, checkpoints BM3 après CV complet). Le dataset compte **~17k phrases** — prévoir un **long run GPU** ; utiliser **`scripts/run_roberta_afd.py`** avec `nohup` ou **tmux** comme pour AFC.

In [ ]:
# AFD — full 35-fold CV + BM3 (binary F1). Very slow compared to AFC.
summary_afd = run_mmused_text_cv(
    get_roberta_afd_config(),
    loader=loader_afd,
    save_bm3_checkpoints_after=True,
)
print(summary_afd)

# Smoke test (uncomment):
# summary_afd = run_mmused_text_cv(
#     get_roberta_afd_config(),
#     loader=loader_afd,
#     max_folds=2,
#     save_bm3_checkpoints_after=False,
# )


### Exécuter AFD en arrière-plan (résiste aux déconnexions SSH)

```bash
cd /mm_argfallacy
mkdir -p logs
nohup ./mmarg_env/bin/python scripts/run_roberta_afd.py > logs/roberta_afd.log 2>&1 &
tail -f logs/roberta_afd.log
```

## AFC & AFD — résultats et interprétation

Après avoir exécuté les deux tâches, `ResultsManager.print_comparison_table()` liste toutes les expériences dans `results/results.json` (macro F1 pour AFC, binary F1 pour AFD).

In [8]:
rm = ResultsManager(str(RESULTS_PATH))
rm.print_comparison_table()



Experiment                Task   Metric              Mean      Std  Folds
roberta_afc               afc    test_macro_f1     0.4489   0.1922     35

Paper baselines
roberta_afc (paper)       afc    macro_f1          0.3925
roberta_afd (paper)       afd    binary_f1         0.2770


RoBERTa-base obtient 0,449 macro F1 (std=0,192) sur 35 folds LODO, soit +5,7 pts au-dessus de la baseline papier (0,393). Mais la variance est l'information principale : std=0,192 signifie que certains folds tombent sous 0,25 et d'autres dépassent 0,90. Ce score moyen agrège des situations très hétérogènes. Tous les résultats multimodaux seront comparés à ce 0,449 comme référence.
